# 05 Qlibによるクオンツ分析

[研究案内](../資料/07_研究コードと成果物の案内.md) / [実装元](../src/quant_ai/qlib_workflow.py)。VS Codeで関数名をF12、または下のセルで実装を表示。

## 1. 目的と仮説
Qlibデータ・式・DatasetH・LightGBM・公式売買評価を一巡する。

## 2. 理論と数式
過去特徴量は$P_t/P_{t-5}-1$等。教師だけに$P_{t+2}/P_{t+1}-1$を使う。

## 3. データの定義
seed42、6合成資産×420営業日、初日価格1に規格化したQlib provider。

## 4. 分析条件・パラメータ
60/20/20時系列分割、2日purge、top2、翌終値、売買各5bps。REG_USは便宜的設定。

## 5. 実装と実行
共通関数の実装も確認してから計算する。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/quant_ai").exists())
os.chdir(ROOT)
(ROOT / "分析結果/図").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("データ種別: 合成データ。実市場での収益性を示す結果ではありません。")


In [ ]:
import inspect
from IPython.display import Code
from quant_ai.qlib_workflow import run_qlib_demo
print(inspect.getsourcefile(run_qlib_demo))
display(Code(inspect.getsource(run_qlib_demo), language='python'))

In [ ]:
from quant_ai.qlib_workflow import build_sample_provider, run_qlib_demo
provider = build_sample_provider(ROOT / "data/qlib_synthetic")
print("provider bytes:", sum(p.stat().st_size for p in provider.rglob("*") if p.is_file()))
result = run_qlib_demo(provider, ROOT / "分析結果/レポート")
display(result["features"].head(12))
display(pd.DataFrame(result["segments"], index=["start", "end"]))
display(result["predictions"].head())


## 6. 計算結果と可視化

In [ ]:
display(result["risk"])
display(result["excess_risk"])
report = result["report"]
print("累積コスト（各日NAV比率の和）:", report.cost.sum())
net = report["return"]-report["cost"]
curves = pd.DataFrame({"strategy_net": (1+net).cumprod(),
                        "benchmark_gross": (1+report.bench).cumprod()})
ax = curves.plot(title="Qlib TopkDropout - synthetic data only")
ax.figure.savefig("分析結果/図/05_Qlibの運用比較.png", dpi=150, bbox_inches="tight")
plt.show()
from quant_ai.metrics import performance
display(pd.DataFrame({"strategy_geometric": performance(net),
                      "benchmark_geometric": performance(report.bench)}))


## 7. 統計的解釈
report.returnは費用前、net=return-cost。Qlib標準年率と幾何CAGRは区別する。

## 8. 金融工学・運用上の解釈
学習成功は実市場での収益性を示さない。同社がQlibを採用しているという前提は置かない。

## 9. 限界と追加検証
## 結果の解釈と約定
TopkDropoutStrategyで上位2資産を選び、翌営業日終値で執行します。
売買それぞれ5bps、最低手数料0、1株単位。Qlibのreport.returnはコスト控除前、
net=return-costで控除します。ベンチマークは合成第1資産の買い持ち（コスト控除前）。
`risk_analysis`の標準年率指標と、独自performanceの幾何CAGRは定義が異なります。
取引日は旧保有の価格変化を受け、新規保有の損益は次期間から発生します。

## 公式データ・変換方法
[公式データ仕様](https://qlib.readthedocs.io/en/stable/component/data.html)、
[v0.9.7 converter](https://github.com/microsoft/qlib/blob/v0.9.7/scripts/dump_bin.py)
を参照。READMEと`資料/02_Qlibの使い方.md`に取得・容量確認・CSV変換手順があります。
公式配布データは容量確認のみで、大規模ダウンロードは行いません。

## 限界・改善
合成データの好成績は実市場での収益性を示しません。固定銘柄集合のため上場廃止・
生存者バイアスは再現しません。実データではpoint-in-time銘柄集合、企業行動、休場、
価格制限、売買代金制約を検証し、市場別設定を明示してください。
US設定はこの教材の便宜的な売買ルールです。日本市場の再現ではありません。
次は公式converterで自分のデータを変換し、時点整合性の監査を行います。